<img src="https://raw.githubusercontent.com/OmarLi-360/excel-corporativo-12-claves/main/assets/logo-drbits.jpg" alt="Dr. Bits" width="170">

# Clave 12 · Auditoría, control y uso responsable de IA
**Excel Corporativo · 12 Claves** — Revisar antes de enviar · ⏱ 30 min

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/OmarLi-360/excel-corporativo-12-claves/blob/main/notebooks/12_auditoria_y_control.ipynb) &nbsp; [Volver al tablero](https://omarli-360.github.io/excel-corporativo-12-claves/#clave-12)

<sub>Si al abrir un enlace Google muestra «Redirect Notice», haz clic en el enlace azul de esa página: es un aviso de seguridad de Google, no un error.</sub>

**Al terminar podrás:**
1. Medir cuánto mueve cada error el total de un informe.
2. Ver cómo cambia la conclusión por sede cuando el informe tiene errores.
3. Probar una fórmula sugerida por una IA con casos conocidos.

---
### Cómo usar este cuaderno (no necesitas programar)
1. **Guarda tu copia:** *Archivo › Guardar una copia en Drive*. Así tus cambios no se pierden. Si no lo haces, no pasa nada grave: el original está protegido y nadie puede dañarlo desde aquí, pero lo que cambies se borra al cerrar.
2. **Ejecuta todo:** *Entorno de ejecución › Ejecutar todas* (Ctrl+F9). Si Colab avisa que el cuaderno no es de Google, elige *Ejecutar de todos modos*.
3. **Juega con los controles:** las celdas con listas desplegables a la derecha se pueden cambiar. Elige una opción y pulsa el botón ▶ de esa celda para redibujar.

El código está oculto a propósito: aquí importa entender el dato y el gráfico. Si tienes curiosidad, haz doble clic en una celda para verlo.

In [ ]:
# @title Preparar el entorno (se ejecuta sola con «Ejecutar todas»)
import os, sys, warnings
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import display, Markdown, HTML
warnings.filterwarnings("ignore")

EN_COLAB = "google.colab" in sys.modules
REPO = "OmarLi-360/excel-corporativo-12-claves"
BASE_URL = f"https://raw.githubusercontent.com/{REPO}/main/data/"
BASE_LOCAL = "../data/"

def ruta(nombre):
    local = os.path.join(BASE_LOCAL, nombre)
    return local if os.path.exists(local) else BASE_URL + nombre

# --- formatos colombianos -------------------------------------------------
def miles(x):    return f"{x:,.0f}".replace(",", ".")
def pesos(x):    return "$ " + miles(x)
def millones(x): return "$ " + f"{x/1e6:,.0f}".replace(",", ".") + " M"
def porc(v, signo=False):
    t = f"{v:+.1%}" if signo else f"{v:.1%}"
    return t.replace(".", ",").replace("%", " %")
MESES = ["Ene", "Feb", "Mar", "Abr", "May", "Jun", "Jul", "Ago", "Sep", "Oct", "Nov", "Dic"]
MM = "Millones de pesos"

# --- plantilla de gráficos Dr. Bits ----------------------------------------
AZUL, TEAL, MAGENTA, GRIS = "#4A5AD8", "#0E9BB8", "#D0336F", "#9AA3BF"
BIEN, MAL = "#15784A", "#B42318"
ejes = dict(gridcolor="#E4E8F2", zeroline=False, linecolor="#C9D0E2", exponentformat="none", separatethousands=True)
pio.templates["drbits"] = go.layout.Template(layout=dict(
    font=dict(family="Source Sans 3, Segoe UI, Arial, sans-serif", size=14, color="#11153A"),
    title=dict(font=dict(size=18, color="#00006A"), x=0, xanchor="left"),
    colorway=[AZUL, TEAL, MAGENTA], plot_bgcolor="white", paper_bgcolor="white",
    xaxis=ejes, yaxis=ejes, separators=",.", hoverlabel=dict(font_size=13),
    margin=dict(t=80, l=10, r=30, b=40)))
pio.templates.default = "plotly_white+drbits"

def eje_meses(fig, fechas, cada=None):
    """Etiquetas de meses en español en ejes de tiempo (Plotly las pone en inglés)."""
    f = pd.to_datetime(pd.Series(sorted(set(pd.to_datetime(list(fechas))))))
    paso = cada or (1 if len(f) <= 13 else 3)
    f = f.iloc[::paso]
    fig.update_xaxes(tickvals=list(f), ticktext=[f"{MESES[d.month - 1]} {d.year}" for d in f])

def dar_aire(fig, maximo, eje="x"):
    (fig.update_xaxes if eje == "x" else fig.update_yaxes)(range=[0, maximo * 1.22])

# --- revisor de retos -----------------------------------------------------
def revisar(respuesta, esperado, explicacion=""):
    if respuesta == "Elige una opción":
        display(Markdown("**Elige una opción en la lista de la derecha** y vuelve a ejecutar esta celda con ▶."))
    elif str(respuesta) == str(esperado):
        display(Markdown(f"✅ **Correcto.** {explicacion}"))
    else:
        display(Markdown(f"❌ **Todavía no.** Revisa el gráfico de arriba e inténtalo otra vez. *(Elegiste: {respuesta})*"))

# --- datos de Distribuidora Andina ----------------------------------------
ventas = pd.read_csv(ruta("ventas.csv"), parse_dates=["fecha"])
clientes = pd.read_csv(ruta("clientes.csv"))
productos = pd.read_csv(ruta("productos.csv"))
vendedores = pd.read_csv(ruta("vendedores.csv"))
sedes = pd.read_csv(ruta("sedes.csv"))
metas = pd.read_csv(ruta("metas.csv"))
df = (ventas
      .merge(sedes[["id_sede", "ciudad"]], on="id_sede", validate="many_to_one")
      .merge(productos[["sku", "producto", "categoria"]], on="sku", validate="many_to_one")
      .merge(clientes[["id_cliente", "razon_social", "segmento"]], on="id_cliente", validate="many_to_one")
      .merge(vendedores[["id_vendedor", "nombre_vendedor"]], on="id_vendedor", validate="many_to_one")
      .assign(anio=lambda d: d.fecha.dt.year, mes=lambda d: d.fecha.dt.month))
validas = df[df["estado"] != "Anulada"]          # las anuladas no son venta
ORDEN_CIUDADES = (validas[validas.anio == 2025].groupby("ciudad")["subtotal"].sum()
                  .sort_values(ascending=False).index.tolist())
display(Markdown(f"Datos listos: **{miles(len(ventas))} líneas de venta**, {len(clientes)} clientes, {len(productos)} productos. "
                 f"Fuente: {'archivos locales' if os.path.exists(BASE_LOCAL) else 'GitHub'}."))

## 1 · El informe con cinco errores
La hoja **Auditoria** de `Andina_listo_para_dinamicas.xlsx` es un informe de cumplimiento 2025 listo para enviar. Tiene cinco errores sembrados:

| Celda | Error | ¿Lo marca Excel? |
|---|---|---|
| B7 | Venta de Cali pegada del informe de 2024 | No |
| B8 | Fórmula sugerida por una IA, sin el criterio de estado: suma anuladas | No |
| C9 | Meta de Bucaramanga escrita como texto: SUMA la ignora | Sí: número como texto |
| D10 | Cumplimiento de Pereira dividido por la meta de la fila de arriba | Sí: fórmula incoherente |
| B11 | El total omite la fila de Pereira | Sí: omite celdas adyacentes |

Este cuaderno reconstruye el informe correcto y el informe tal como quedó, para ver el efecto de cada error.

In [ ]:
# @title Reconstruir el informe correcto y el reportado
v25 = validas[validas.anio == 2025].groupby("ciudad")["subtotal"].sum()
v25_con_anuladas = df[df.anio == 2025].groupby("ciudad")["subtotal"].sum()
v24 = validas[validas.anio == 2024].groupby("ciudad")["subtotal"].sum()
meta = metas[metas.anio == 2025].merge(sedes[["id_sede", "ciudad"]], on="id_sede").groupby("ciudad")["meta_subtotal"].sum()
orden = ["Bogotá", "Medellín", "Cali", "Barranquilla", "Bucaramanga", "Pereira"]
ok = pd.DataFrame({"venta": v25, "meta": meta}).reindex(orden)
ok["cumplimiento"] = ok.venta / ok.meta
rep = ok.copy()
rep.loc["Cali", "venta"] = v24["Cali"]                                   # B7
rep.loc["Barranquilla", "venta"] = v25_con_anuladas["Barranquilla"]       # B8
rep["cumplimiento"] = rep.venta / rep.meta
rep.loc["Pereira", "cumplimiento"] = rep.loc["Pereira", "venta"] / rep.loc["Bucaramanga", "meta"]   # D10
total_ok = ok.venta.sum()
total_rep = rep.venta.drop("Pereira").sum()                              # B11
meta_rep = rep.meta.drop("Bucaramanga").sum()                            # C9
display(Markdown(f"Total correcto: **{pesos(total_ok)}** · Total reportado: **{pesos(total_rep)}** · Diferencia: **{'−' if total_rep < total_ok else '+'}{pesos(abs(total_rep - total_ok))}**"))

> **Ficha de decisión**
>
> | | |
> |---|---|
> | **Pregunta** | ¿Cuánto mueve cada error el total de venta? |
> | **Qué se grafica** | Cascada: del total correcto al total reportado, un escalón por error |
> | **Por qué esta forma** | La cascada explica una diferencia como suma de partes: cada escalón es un error y su tamaño. |
> | **Qué se descartó** | Solo la diferencia total: no se sabe qué corregir primero. |
> | **Cómo se lee** | Los escalones hacia abajo restan venta; los de arriba la suman. El más largo es el error más costoso. |
> | **En Excel 365** | Filas 14 y 15 de la hoja Auditoria: el cuadre contra la fuente muestra la diferencia total. |

In [ ]:
# @title Gráfico 1 · Del total correcto al reportado
efectos = pd.Series({
    "Cali: valor pegado de 2024": v24["Cali"] - v25["Cali"],
    "Barranquilla: fórmula de IA sin criterio de estado": v25_con_anuladas["Barranquilla"] - v25["Barranquilla"],
    "Total: omite Pereira": -v25["Pereira"],
})
fig = go.Figure(go.Waterfall(
    x=["Total correcto", *[e.split(":")[0] + ":<br>" + e.split(": ")[1] for e in efectos.index], "Total reportado"],
    y=[total_ok / 1e6, *(efectos / 1e6).tolist(), 0], measure=["absolute", "relative", "relative", "relative", "total"],
    text=[f"{v:,.0f}".replace(",", ".") for v in [total_ok / 1e6, *(efectos / 1e6).tolist(), total_rep / 1e6]], textposition="outside",
    increasing=dict(marker_color=MAGENTA), decreasing=dict(marker_color=MAGENTA), totals=dict(marker_color=AZUL),
    connector=dict(line=dict(color=GRIS))))
fig.update_layout(title=f"El informe reporta {millones(abs(total_rep - total_ok))} menos que la base; el error más costoso es «{efectos.abs().idxmax()}»",
                  yaxis_title=MM, height=420, yaxis_range=[0, total_ok / 1e6 * 1.15])
fig.show()

## 2 · La conclusión que cambia

> **Ficha de decisión**
>
> | | |
> |---|---|
> | **Pregunta** | ¿Cambia el mensaje por sede cuando el informe tiene errores? |
> | **Qué se grafica** | Puntos conectados: cumplimiento correcto y reportado por sede · Línea en 100 % |
> | **Por qué esta forma** | Cada línea une los dos valores de una sede: cuanto más larga, más cambia la conclusión. |
> | **Qué se descartó** | Dos tablas de cumplimiento: habría que comparar cifra por cifra. |
> | **Cómo se lee** | Pereira pasa de cumplir a no cumplir, y Cali parece mucho más lejos de su meta de lo que está. |
> | **En Excel 365** | Columna D de la hoja Auditoria antes y después de corregir. |

In [ ]:
# @title Gráfico 2 · Cumplimiento correcto frente al reportado
fig = go.Figure()
for s in orden:
    a, b = ok.cumplimiento[s], rep.cumplimiento[s]
    fig.add_scatter(x=[a, b], y=[s, s], mode="lines", line=dict(color=GRIS, width=2), showlegend=False)
fig.add_scatter(x=ok.cumplimiento, y=orden, mode="markers", name="Correcto", marker=dict(color=AZUL, size=12))
fig.add_scatter(x=rep.cumplimiento, y=orden, mode="markers", name="Reportado", marker=dict(color=MAGENTA, size=12, symbol="diamond"))
fig.add_vline(x=1, line_dash="dash", line_color="#11153A", annotation_text="Meta 100 %")
cambian = [s for s in orden if (ok.cumplimiento[s] >= 1) != (rep.cumplimiento[s] >= 1)]
fig.update_layout(title=f"Con los errores, {', '.join(cambian) if cambian else 'ninguna sede'} cambia de conclusión frente a la meta",
                  xaxis_tickformat=".0%", height=360, yaxis=dict(autorange="reversed"), legend=dict(orientation="h", y=-0.2))
fig.show()

## 3 · Probar la fórmula de la IA con casos conocidos
La fórmula de B8 la sugirió un asistente de IA. Se veía razonable y nadie la probó. La prueba es simple: calcular el mismo dato por otro camino (una tabla dinámica sin anuladas) en varios casos y comparar.

> **Ficha de decisión**
>
> | | |
> |---|---|
> | **Pregunta** | ¿En qué meses falla la fórmula sugerida? |
> | **Qué se grafica** | Columnas por mes: diferencia entre la fórmula de la IA y la tabla dinámica para la sede elegida |
> | **Por qué esta forma** | Cada mes es un caso de prueba; una columna distinta de cero es un caso fallido. |
> | **Qué se descartó** | Probar solo el total anual: un error pequeño en un mes se esconde en el total. |
> | **Cómo se lee** | Los meses con columna son los que tienen facturas anuladas: la IA no las excluyó. |
> | **En Excel 365** | Tabla dinámica de tVentas con ciudad y mes, sin anuladas, comparada con la fórmula en cada mes. |

In [ ]:
# @title Gráfico 3 · Casos de prueba mes a mes
sede_prueba = "Barranquilla"  # @param ["Bogotá", "Medellín", "Cali", "Barranquilla", "Bucaramanga", "Pereira"]
ia = df[(df.anio == 2025) & (df.ciudad == sede_prueba)].groupby("mes")["subtotal"].sum()
dinamica = validas[(validas.anio == 2025) & (validas.ciudad == sede_prueba)].groupby("mes")["subtotal"].sum()
dif = (ia - dinamica).reindex(range(1, 13), fill_value=0) / 1e6
fallan = int((dif > 0).sum())
fig = go.Figure(go.Bar(x=[MESES[m - 1] for m in dif.index], y=dif.values, marker_color=[MAGENTA if v > 0 else GRIS for v in dif.values],
                       text=[f"{v:,.1f}".replace(".", ",") if v > 0 else "" for v in dif.values], textposition="outside"))
fig.update_layout(title=f"{sede_prueba}: la fórmula de la IA falla en {fallan} de 12 meses (suma facturas anuladas)",
                  yaxis_title="Diferencia, millones", height=340)
fig.show()

## 4 · Retos de interpretación

In [ ]:
# @title Reto 1 · ¿Qué error mueve más el total de venta del informe?
respuesta = "Elige una opción"  # @param ["Elige una opción", "Cali: valor pegado de 2024", "Barranquilla: fórmula de IA sin criterio de estado", "Total: omite Pereira"]
revisar(respuesta, efectos.abs().idxmax(), """Es el escalón más largo de la cascada.""")

In [ ]:
# @title Reto 2 · ¿Cuántos de los cinco errores marca la Comprobación de errores de Excel?
respuesta = "Elige una opción"  # @param ["Elige una opción", "1", "3", "5", "0"]
revisar(respuesta, "3", """Marca el número como texto, la fórmula incoherente y el total que omite celdas. El valor pegado y la fórmula de la IA no tienen aviso.""")

In [ ]:
# @title Reto 3 · Una IA te sugiere una fórmula y el resultado se ve razonable. ¿Qué haces antes de enviarlo?
respuesta = "Elige una opción"  # @param ["Elige una opción", "Enviarlo: se ve bien", "Probarla con casos conocidos calculados por otro camino", "Pedirle a la IA que confirme", "Agregar SI.ERROR"]
revisar(respuesta, "Probarla con casos conocidos calculados por otro camino", """Un resultado razonable puede estar mal. La prueba con casos conocidos lo demuestra.""")

## Apóyate en IA
Para diseñar los casos de prueba de una fórmula:

```text
Una IA me sugirió esta fórmula de Excel 365: [PEGA LA FÓRMULA].
Debe calcular la venta neta por sede y año, sin facturas anuladas.
Propón tres casos de prueba que pueda verificar con una tabla dinámica, incluido uno que detecte si suma anuladas.
```

> **Verifica siempre.** Compara la respuesta de la IA con un dato que ya conozcas. No pegues datos personales reales (Ley 1581 de 2012): estos datos son ficticios, los de tu empresa no.

## Resumen
* Mostrar fórmulas y la comprobación de errores encuentran parte de los errores, no todos.
* El cuadre contra la fuente y la prueba con casos conocidos encuentran el resto.
* Lo que propone una IA se prueba igual que lo que escribe un colega: lo que envías es tu responsabilidad.

**Siguiente:** [Laboratorio Excel + Colab](https://omarli-360.github.io/excel-corporativo-12-claves/#laboratorio) · [Taller 3](https://colab.research.google.com/github/OmarLi-360/excel-corporativo-12-claves/blob/main/notebooks/taller_3_tablero_comite.ipynb)

<sub>Dr. Bits · ADNBITS S.A.S. · Contenido CC BY-SA 4.0 · Datos ficticios de Distribuidora Andina S.A.S.</sub>